[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johanhoffman/DD2365_FEniCSx/blob/main/verification/schafer-turek-2d1.ipynb)

# **Schäfer-Turek 2D-1 benchmark**
**Johan Hoffman**

# **Abstract**

The GLS stabilized P1/P1 Navier-Stokes solver is applied to the Schäfer-Turek 2D-1 benchmark (steady Re=20 flow past a cylinder in a channel, Schäfer & Turek 1996) using a graded mesh (gmsh_rect_minus_circles v6, dist_min=0.5D=0.05 m, dist_max=3D=0.30 m). Three refinement levels are targeted; levels 1 and 2 are complete, level 3 (resolution=128) is running. Force is computed via the volume form (Green's formula, primary); surface stress results are a secondary column. Steady state is reached at t ≈ 19.0 (L1) and t ≈ 19.4 (L2). C_D converges at rate ≈1.0 (L1: 0.40% error, L2: 0.20% error), just above the ST96 upper bound at L2 (5.5908 vs [5.57, 5.59]). C_L recovers the correct sign and enters the ST96 interval at L2 (0.01077 ✓ in [0.0104, 0.0110]); at L1 the wake is under-resolved and C_L = −0.0074. Pressure drop Δp shows non-monotone convergence: L1 overshoots (0.11811) and L2 undershoots (0.11653) the reference 0.11752.

# **About the code**

In [ ]:
# Copyright (C) 2020-2026 Johan Hoffman (jhoffman@kth.se)
# SPDX-License-Identifier: MIT
#
# Verification notebook: Schäfer-Turek 2D-1 steady benchmark.
# This template is maintained by Johan Hoffman.
# Please report problems to jhoffman@kth.se

In [ ]:
# --- canonical: bootstrap v2 ---
import os as _os
_os.environ.setdefault("OMP_NUM_THREADS", "1")
_os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
_os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
_os.environ.setdefault("MKL_NUM_THREADS", "1")

import sys, os, subprocess

_on_colab = "google.colab" in sys.modules

if _on_colab:
    try:
        import gmsh
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/gmsh-install.sh"'
            ' -O /tmp/gmsh-install.sh && bash /tmp/gmsh-install.sh',
            shell=True, check=True,
        )
    try:
        import dolfinx
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh"'
            ' -O /tmp/fenicsx-install.sh && bash /tmp/fenicsx-install.sh',
            shell=True, check=True,
        )

import dolfinx
print("dolfinx version:", dolfinx.__version__)

if _on_colab:
    from google.colab import files
# --- end canonical: bootstrap ---

In [ ]:
import numpy as np
import time
import ufl
import basix.ufl as bufl
from mpi4py import MPI
from matplotlib import pyplot as plt

import dolfinx
from dolfinx import fem
from dolfinx.fem import functionspace, Function, form, assemble_scalar, Constant
from dolfinx.fem.petsc import (
    assemble_matrix, assemble_vector, apply_lifting, set_bc,
    create_matrix, create_vector,
)
from petsc4py import PETSc

In [ ]:
# --- canonical: gmsh_rect_minus_circles v6 ---
import math
import numpy as np
import gmsh
from mpi4py import MPI
from dolfinx.io import gmsh as gmshio


_ALPHA = 0.66
# mshr numSamples=100 → N = floor(sqrt(100.5)) = 10 interior sample rows
_N_SAMPLE = int(math.floor(math.sqrt(100.0 + 0.5)))  # 10


def _mshr_segments(r, L, H, resolution):
    """Exact mshr auto-segment count for Circle(segments=0) in 2D.

    Source path:
      MeshGenerator.cpp: segment_granularity = 2*R_enc/resolution
      CSGGeometry.cpp:   estimate_bounding_sphere samples _N_SAMPLE×_N_SAMPLE
                         interior points; extreme corners dominate, giving
                         R_enc = ((N-1)/N) * sqrt(L²+H²) / 2
      CSGCGALDomain2D.cpp: num_segs = max(5, round(2π*r / segment_granularity))

    Verified predictions (no Steiner points since edge_len < cs_mshr):
      4×2 domain, r=0.2, resolution=32 → 10
      4×4 domain, r=0.2, resolution=32 →  8
    """
    R_enc = ((_N_SAMPLE - 1) / _N_SAMPLE) * math.sqrt(L ** 2 + H ** 2) / 2.0
    cs = 2.0 * R_enc / resolution
    return max(5, round(2.0 * math.pi * r / cs))


def gmsh_rect_minus_circles(L, H, circles, resolution, segments=32,
                             dist_min=None, dist_max=None):
    """Rectangle [0,L]×[0,H] minus regular-polygon holes.

    Parameters
    ----------
    L, H : float
        Rectangle dimensions.
    circles : list of (cx, cy, r)
        Circle centres and radii.
    resolution : int
        Mesh density parameter.  Interior mesh size:
            lc = _ALPHA * sqrt(L² + H²) / resolution
    segments : int or "mshr"
        Number of polygon sides used to approximate each circle hole.
        Default 32.  Pass ``"mshr"`` to use the legacy mshr auto-segment
        rule (max(5, round(2π·r/cs)), cs derived from bounding-sphere
        estimate), which gives 10 sides for 4×2 domains and 8 for 4×4.
    dist_min : float or None
        Threshold DistMin per hole (inner boundary of transition zone).
        Default None → 0.0 (fine zone starts at the polygon surface).
    dist_max : float or None
        Threshold DistMax per hole (outer boundary of transition zone).
        Default None → r/2 per hole (reproduces v5 behaviour exactly).

    Returns
    -------
    msh : dolfinx.mesh.Mesh
    cell_tags : dolfinx.mesh.MeshTags   (fluid domain tag = 10)

    Notes
    -----
    Vertex placement: phi_i = 2π·i/n, i = 0 … n-1 (i=0 at angle 0, CCW).

    Mesh sizing: one Distance+Threshold gmsh field per polygon transitions
    from ``edge_len = 2r·sin(π/n)`` right at the polygon boundary (prevents
    Steiner-point insertion on polygon edges) to ``lc`` at distance
    ``dist_max`` (default r/2).  No MeshSizeMax override; background field
    is the sole size control.
    """
    lc = _ALPHA * math.sqrt(L ** 2 + H ** 2) / resolution
    eps = 1e-6 * max(L, H)

    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 0)

    gmsh.model.occ.addRectangle(0, 0, 0, L, H, tag=1)

    hole_tags = []
    segs_per_circle = []
    for cx, cy, r in circles:
        if segments == "mshr":
            n = _mshr_segments(r, L, H, resolution)
        elif isinstance(segments, int):
            n = segments
        else:
            raise ValueError(f"segments must be an int or 'mshr', got {segments!r}")
        segs_per_circle.append(n)
        pts = [
            gmsh.model.occ.addPoint(
                cx + r * math.cos(2 * math.pi * i / n),
                cy + r * math.sin(2 * math.pi * i / n),
                0,
            )
            for i in range(n)
        ]
        lines = [
            gmsh.model.occ.addLine(pts[i], pts[(i + 1) % n])
            for i in range(n)
        ]
        cl = gmsh.model.occ.addCurveLoop(lines)
        surf = gmsh.model.occ.addPlaneSurface([cl])
        hole_tags.append((2, surf))

    if hole_tags:
        gmsh.model.occ.cut([(2, 1)], hole_tags)
    gmsh.model.occ.synchronize()

    # Distance+Threshold field per polygon hole
    bg_fields = []
    for i, (cx, cy, r) in enumerate(circles):
        n = segs_per_circle[i]
        edge_len = 2 * r * math.sin(math.pi / n)

        # Identify polygon curves (not on the rectangle boundary)
        c_curves = []
        for dim, tag in gmsh.model.getEntities(1):
            xmin, ymin, _, xmax, ymax, _ = gmsh.model.getBoundingBox(dim, tag)
            if xmin <= eps or xmax >= L - eps or ymin <= eps or ymax >= H - eps:
                continue
            bounds = gmsh.model.getParametrizationBounds(dim, tag)
            mid = float(np.array(bounds).mean())
            x, y, _ = gmsh.model.getValue(dim, tag, [mid])
            if abs(math.hypot(x - cx, y - cy) - r) < 0.15 * r:
                c_curves.append(tag)

        if not c_curves:
            continue

        dm_min = 0.0 if dist_min is None else dist_min
        dm_max = r / 2.0 if dist_max is None else dist_max

        d_id = 10 + 2 * i
        t_id = 11 + 2 * i
        gmsh.model.mesh.field.add("Distance", d_id)
        gmsh.model.mesh.field.setNumbers(d_id, "CurvesList", c_curves)
        gmsh.model.mesh.field.add("Threshold", t_id)
        gmsh.model.mesh.field.setNumber(t_id, "InField", d_id)
        # SizeMin = edge_len at polygon: prevents subdivision of polygon edges.
        # SizeMax = lc at dist_max: interior mesh size.
        gmsh.model.mesh.field.setNumber(t_id, "SizeMin", edge_len)
        gmsh.model.mesh.field.setNumber(t_id, "SizeMax", lc)
        gmsh.model.mesh.field.setNumber(t_id, "DistMin", dm_min)
        gmsh.model.mesh.field.setNumber(t_id, "DistMax", dm_max)
        bg_fields.append(t_id)

    if bg_fields:
        if len(bg_fields) == 1:
            gmsh.model.mesh.field.setAsBackgroundMesh(bg_fields[0])
        else:
            min_id = 10 + 2 * len(circles)
            gmsh.model.mesh.field.add("Min", min_id)
            gmsh.model.mesh.field.setNumbers(min_id, "FieldsList", bg_fields)
            gmsh.model.mesh.field.setAsBackgroundMesh(min_id)

    # Background field is the sole size control
    gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)
    gmsh.option.setNumber("Mesh.Algorithm", 5)  # Delaunay

    surfaces = gmsh.model.getEntities(2)
    gmsh.model.addPhysicalGroup(2, [s[1] for s in surfaces], tag=10, name="domain")
    gmsh.model.mesh.generate(2)

    mesh_data = gmshio.model_to_mesh(gmsh.model, MPI.COMM_WORLD, rank=0, gdim=2)
    gmsh.finalize()
    return mesh_data.mesh, mesh_data.cell_tags
# --- end canonical: gmsh_rect_minus_circles ---

In [ ]:
# --- canonical: tag_boundaries v1 ---
import numpy as np
from dolfinx.mesh import locate_entities_boundary, meshtags


def tag_boundaries(msh, L, H, eps=None):
    """Tag exterior boundary facets of a [0,L]×[0,H] rectangle with holes.

    Assigns integer tags to all exterior boundary facets:

        left=1  (x ≈ 0),   right=2 (x ≈ L),
        lower=3 (y ≈ 0),   upper=4 (y ≈ H),
        objects=5 (remaining exterior facets — circle boundaries).

    Corners are unambiguous: boundary facets are edges, not vertices, so a
    corner vertex is shared by one vertical and one horizontal edge — each
    edge belongs to exactly one group and no facet is tagged twice.

    Parameters
    ----------
    msh : dolfinx.mesh.Mesh
    L, H : float
        Rectangle dimensions.
    eps : float, optional
        Coordinate tolerance.  Default ``1e-6 * max(L, H)``.

    Returns
    -------
    dolfinx.mesh.MeshTags
        Must be recomputed whenever the mesh changes (e.g. after refinement).
    """
    if eps is None:
        eps = 1e-6 * max(L, H)

    fdim = msh.topology.dim - 1
    msh.topology.create_entities(fdim)
    msh.topology.create_connectivity(fdim, msh.topology.dim)

    left  = locate_entities_boundary(msh, fdim, lambda x: x[0] <= eps)
    right = locate_entities_boundary(msh, fdim, lambda x: x[0] >= L - eps)
    lower = locate_entities_boundary(msh, fdim, lambda x: x[1] <= eps)
    upper = locate_entities_boundary(msh, fdim, lambda x: x[1] >= H - eps)

    known = np.unique(np.concatenate([left, right, lower, upper]))
    all_bdry = locate_entities_boundary(
        msh, fdim, lambda x: np.ones(x.shape[1], dtype=bool)
    )
    objects = np.setdiff1d(all_bdry, known)

    indices = np.concatenate([left, right, lower, upper, objects]).astype(np.int32)
    values  = np.concatenate([
        np.full(len(left),    1, dtype=np.int32),
        np.full(len(right),   2, dtype=np.int32),
        np.full(len(lower),   3, dtype=np.int32),
        np.full(len(upper),   4, dtype=np.int32),
        np.full(len(objects), 5, dtype=np.int32),
    ])
    order = np.argsort(indices)
    return meshtags(msh, fdim, indices[order], values[order])
# --- end canonical: tag_boundaries ---

In [ ]:
# --- canonical: plot_helpers v3 ---
import numpy as np
import matplotlib.pyplot as plt
import basix.ufl as _bufl
from dolfinx.fem import functionspace, Function


def plot_mesh(msh, title="Mesh"):
    """Plot a 2-D triangular mesh using matplotlib triplot."""
    msh.topology.create_connectivity(msh.topology.dim, 0)
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.triplot(x[:, 0], x[:, 1], gdm, linewidth=0.3, color="k")
    ax.set_aspect("equal")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def _p1_scalar_space(msh):
    return functionspace(msh, ("Lagrange", 1))


def _scatter_p1_scalar(f1):
    """Scatter a P1 scalar Function values to geometry nodes.

    Returns (x, geometry_connectivity, node_values).
    """
    V = f1.function_space
    msh = V.mesh
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    ldm = V.dofmap.list
    vals = np.zeros(x.shape[0])
    vals[gdm.ravel()] = f1.x.array.real[ldm.ravel()]
    return x, gdm, vals


def plot_p1(u, title="Solution"):
    """Plot a P1 scalar Function using matplotlib tripcolor (Gouraud shading)."""
    plot_scalar(u, title=title)


def plot_scalar(f, title="Scalar field", ax=None):
    """Plot any scalar Lagrange Function via tripcolor (interpolates to P1 if needed).

    Parameters
    ----------
    f : dolfinx.fem.Function   scalar; any Lagrange degree
    title : str
    ax : matplotlib.axes.Axes or None
        If given, draw into this axes (no new figure, no plt.show()).
        If None (default), create a new figure and call plt.show().
    """
    V = f.function_space
    msh = V.mesh
    el = V.ufl_element()
    if el.degree == 1 and el.reference_value_shape == ():
        f1 = f
    else:
        f1 = Function(_p1_scalar_space(msh))
        f1.interpolate(f)
    x, gdm, vals = _scatter_p1_scalar(f1)
    _standalone = ax is None
    if _standalone:
        fig, ax = plt.subplots(figsize=(8, 3))
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, vals, shading="gouraud")
    plt.colorbar(tc, ax=ax)
    ax.set_aspect("equal")
    ax.set_title(title)
    if _standalone:
        plt.tight_layout()
        plt.show()


def plot_vector(u, title="Vector field", quiver=True, quiver_stride=8, ax=None):
    """Plot a 2-D vector Lagrange Function as colour map of |u| + optional quiver.

    Parameters
    ----------
    u : dolfinx.fem.Function   value shape (2,); any Lagrange degree
    title : str
    quiver : bool              overlay subsampled arrows (default True)
    quiver_stride : int        take every N-th mesh node for arrows
    ax : matplotlib.axes.Axes or None
        If given, draw into this axes (no new figure, no plt.show()).
        If None (default), create a new figure and call plt.show().
    """
    msh = u.function_space.mesh
    gdim = msh.geometry.dim
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    npts = x.shape[0]

    # Interpolate to P1 vector so values align with geometry nodes
    V1v = functionspace(msh, _bufl.element("Lagrange", msh.basix_cell(), 1, shape=(gdim,)))
    u1v = Function(V1v)
    u1v.interpolate(u)

    # Scatter: for a block-size-2 P1 space, dofmap.list gives block indices
    ldm = V1v.dofmap.list      # (ncells, 3)  — block indices
    arr = u1v.x.array.real     # length npts * gdim, interleaved per block
    ux = np.zeros(npts)
    uy = np.zeros(npts)
    ux[gdm.ravel()] = arr[ldm.ravel() * gdim + 0]
    uy[gdm.ravel()] = arr[ldm.ravel() * gdim + 1]
    mag = np.sqrt(ux**2 + uy**2)

    _standalone = ax is None
    if _standalone:
        fig, ax = plt.subplots(figsize=(8, 3))
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, mag, shading="gouraud", cmap="viridis")
    plt.colorbar(tc, ax=ax, label="|u|")
    if quiver:
        idx = np.arange(0, npts, quiver_stride)
        ax.quiver(x[idx, 0], x[idx, 1], ux[idx], uy[idx],
                  color="white", alpha=0.6, width=0.002, scale_units="xy", scale=2.0)
    ax.set_aspect("equal")
    ax.set_title(title)
    if _standalone:
        plt.tight_layout()
        plt.show()
# --- end canonical: plot_helpers ---

# **Introduction**

The Schäfer-Turek benchmark describes steady laminar flow past a circular cylinder in a channel. The domain is [0, 2.2] × [0, 0.41] with a cylinder of radius r = 0.05 centred at (0.2, 0.2). A parabolic inflow profile is prescribed at the left boundary. Reference quantities (Schäfer & Turek 1996, 2D-1 steady): C_D = 5.57953523384, C_L = 0.010618948146, pressure drop Δp = 0.11752016697. Accepted ST96 intervals: C_D ∈ [5.57, 5.59], C_L ∈ [0.0104, 0.0110], Δp ∈ [0.1172, 0.1176].

Three mesh refinement levels (resolution = 32, 64, 128; segments = 32, 64, 128 per circle) are used to assess convergence. A graded mesh (gmsh_rect_minus_circles v6) concentrates resolution near the cylinder: the Threshold size field holds SizeMin = edge_len = 2r sin(π/n) from the cylinder surface to dist_min = 0.5 D = 0.05 m, then transitions linearly to the background size lc at dist_max = 3 D = 0.30 m. D = 2r = 0.1 m is the cylinder diameter. This extends the fine-mesh zone well into the cylinder wake, where the near-cancellation that makes C_L sensitive to resolution is concentrated.

# **Method**

**Parameters**

In [ ]:
# Problem geometry
L, H = 2.2, 0.41
cx, cy, r = 0.2, 0.2, 0.05
D = 2.0 * r           # cylinder diameter = 0.1 m

# Physical parameters
nu = 1e-3             # kinematic viscosity
U_m = 0.3             # max inflow velocity; mean U = 2/3·U_m

# Simulation time and convergence
T = 30.0              # max integration time
plot_freq = 5
tol_ss = 1e-6         # steady-state criterion: ||u1-u0||/dt/||u1|| < tol_ss

# Graded mesh parameters (v6): fine zone holds SizeMin to dist_min, then
# transitions to background lc at dist_max.
dist_min_grade = 0.5 * D   # 0.05 m — fine zone to 0.5 D from cylinder surface
dist_max_grade = 3.0 * D   # 0.30 m — transition to lc completes at 3 D

# Reference values (Schäfer & Turek 1996, 2D-1 steady benchmark)
C_D_ref = 5.57953523384
C_L_ref = 0.010618948146
dp_ref  = 0.11752016697

# Refinement levels: coarse, medium, fine
levels = [
    {"resolution": 32,  "segments": 32},
    {"resolution": 64,  "segments": 64},
    {"resolution": 128, "segments": 128},
]

**Convergence study**

In [ ]:
from dolfinx.geometry import (
    bb_tree as _bb_tree,
    compute_collisions_points as _ccp,
    compute_colliding_cells as _ccc,
)

U_mean    = (2.0 / 3.0) * U_m
force_ref = -2.0 / (U_mean**2 * D)   # C = force_ref * F_form (rho=1)

results = []

for lvl_idx, lvl in enumerate(levels):
    if lvl_idx > 0 and T < 1.0:   # fast mode: coarsest level only
        results.append(None)
        continue

    resolution = lvl["resolution"]
    segs       = lvl["segments"]
    print(f"\n--- Level {lvl_idx + 1}: resolution={resolution}, segments={segs} ---",
          flush=True)

    # Graded mesh (v6): fine zone [0, dist_min_grade], transition to lc at dist_max_grade
    msh, _ = gmsh_rect_minus_circles(
        L, H, [(cx, cy, r)], resolution, segments=segs,
        dist_min=dist_min_grade, dist_max=dist_max_grade,
    )
    facet_tags = tag_boundaries(msh, L, H)

    tdim    = msh.topology.dim
    fdim    = tdim - 1
    n_cells = msh.topology.index_map(tdim).size_local
    h_arr   = msh.h(tdim, np.arange(n_cells, dtype=np.int32))

    # h at cylinder: min cell size among cells adjacent to cylinder facets
    msh.topology.create_connectivity(fdim, tdim)
    f2c      = msh.topology.connectivity(fdim, tdim)
    cyl_f    = facet_tags.find(5)
    cyl_adj  = np.unique(np.concatenate([f2c.links(f) for f in cyl_f]))
    h_cyl    = float(msh.h(tdim, cyl_adj.astype(np.int32)).min())

    # h at distance D from cylinder center: mean cell size in annular band [0.8D, 1.2D]
    msh.topology.create_connectivity(tdim, 0)
    c2v      = msh.topology.connectivity(tdim, 0)
    all_ci   = np.arange(n_cells, dtype=np.int32)
    xg       = msh.geometry.x
    ctrs     = np.array([xg[c2v.links(c)].mean(axis=0) for c in all_ci])
    d_from_c = np.sqrt((ctrs[:,0]-cx)**2 + (ctrs[:,1]-cy)**2)
    band     = (d_from_c >= 0.8*D) & (d_from_c <= 1.2*D)
    h_D_val  = float(h_arr[band].mean()) if band.any() else float('nan')

    print(f"  cells={n_cells}, h_cyl={h_cyl:.5f}, h_D={h_D_val:.5f}", flush=True)

    plot_mesh(msh, title=f"Level {lvl_idx + 1}: res={resolution}, cells={n_cells}")

    # Spaces
    P1v = bufl.element("Lagrange", msh.basix_cell(), 1, shape=(2,))
    P1s = bufl.element("Lagrange", msh.basix_cell(), 1)
    V   = functionspace(msh, P1v)
    Q   = functionspace(msh, P1s)
    V0, _ = V.sub(0).collapse()
    V1, _ = V.sub(1).collapse()

    inlet_facets  = facet_tags.find(1)
    cyl_facets    = facet_tags.find(5)
    wall_facets   = np.concatenate([facet_tags.find(3), facet_tags.find(4)])
    outlet_facets = facet_tags.find(2)

    def _zero_bc(sub_idx, sub_V, facets):
        f = Function(sub_V); f.x.array[:] = 0.0
        dofs = fem.locate_dofs_topological((V.sub(sub_idx), sub_V), fdim, facets)
        return fem.dirichletbc(f, dofs, V.sub(sub_idx))

    u_in_func = Function(V)
    u_in_func.interpolate(lambda x: np.vstack([
        4.0 * U_m * x[1] * (H - x[1]) / H**2,
        np.zeros_like(x[1]),
    ]))
    bcu_in   = fem.dirichletbc(u_in_func,
                               fem.locate_dofs_topological(V, fdim, inlet_facets))
    bcu_wall = [_zero_bc(0, V0, wall_facets), _zero_bc(1, V1, wall_facets)]
    bcu_cyl  = [_zero_bc(0, V0, cyl_facets),  _zero_bc(1, V1, cyl_facets)]
    bcu = [bcu_in] + bcu_wall + bcu_cyl
    bcp = [fem.dirichletbc(Constant(msh, PETSc.ScalarType(0.0)),
                           fem.locate_dofs_topological(Q, fdim, outlet_facets), Q)]

    u0 = Function(V, name="u0"); u1 = Function(V, name="u1")
    p1 = Function(Q, name="p1")
    dt_val = 0.5 * h_cyl   # dt based on cylinder mesh size
    dt_c   = Constant(msh, PETSc.ScalarType(dt_val))
    nu_c   = Constant(msh, PETSc.ScalarType(nu))

    u = ufl.TrialFunction(V); v = ufl.TestFunction(V)
    p = ufl.TrialFunction(Q); q = ufl.TestFunction(Q)

    h_c   = ufl.CellDiameter(msh)
    u_mag = ufl.sqrt(ufl.dot(u1, u1))
    d1    = 1.0 / ufl.sqrt((1.0/dt_c)**2 + (u_mag/h_c)**2)
    d2    = h_c * u_mag
    um    = 0.5 * (u + u0)
    um1   = 0.5 * (u1 + u0)

    Fu = (ufl.inner((u - u0)/dt_c + ufl.dot(ufl.grad(um), um1), v)*ufl.dx
        - p1*ufl.div(v)*ufl.dx
        + nu_c*ufl.inner(ufl.grad(um), ufl.grad(v))*ufl.dx
        + d1*ufl.inner((u - u0)/dt_c + ufl.dot(ufl.grad(um), um1) + ufl.grad(p1),
                       ufl.dot(ufl.grad(v), um1))*ufl.dx
        + d2*ufl.div(um)*ufl.div(v)*ufl.dx)

    Fp = (d1*ufl.inner((u1 - u0)/dt_c + ufl.dot(ufl.grad(um1), um1) + ufl.grad(p),
                       ufl.grad(q))*ufl.dx
        + ufl.div(um1)*q*ufl.dx)

    a_u = form(ufl.lhs(Fu)); l_u = form(ufl.rhs(Fu))
    a_p = form(ufl.lhs(Fp)); l_p = form(ufl.rhs(Fp))

    # Volume-form force (primary — Green's formula, course method)
    def _make_psi(phi_x, phi_y):
        psi = Function(V); psi.x.array[:] = 0.0
        dx_dofs = fem.locate_dofs_topological((V.sub(0), V0), fdim, cyl_facets)
        dy_dofs = fem.locate_dofs_topological((V.sub(1), V1), fdim, cyl_facets)
        if len(dx_dofs): psi.x.array[dx_dofs[0]] = phi_x
        if len(dy_dofs): psi.x.array[dy_dofs[0]] = phi_y
        psi.x.scatter_forward()
        return psi

    def _make_force_form(psi_f):
        return form(
            ufl.inner((u1 - u0)/dt_c + ufl.dot(ufl.grad(um1), um1), psi_f)*ufl.dx
            - p1*ufl.div(psi_f)*ufl.dx
            + nu_c*ufl.inner(ufl.grad(um1), ufl.grad(psi_f))*ufl.dx
        )

    psi_drag  = _make_psi(1.0, 0.0)
    psi_lift  = _make_psi(0.0, 1.0)
    drag_form = _make_force_form(psi_drag)
    lift_form = _make_force_form(psi_lift)

    # Surface stress force (secondary — ∫(σ·n)·e dS on cylinder; authorized here only)
    n_vec    = ufl.FacetNormal(msh)
    ds_cyl   = ufl.Measure("ds", domain=msh, subdomain_data=facet_tags, subdomain_id=5)
    sigma_s  = -p1 * ufl.Identity(2) + nu_c * (ufl.grad(u1) + ufl.grad(u1).T)
    surf_drag_form = form(ufl.dot(ufl.dot(sigma_s, n_vec), ufl.as_vector([1.0, 0.0])) * ds_cyl)
    surf_lift_form = form(ufl.dot(ufl.dot(sigma_s, n_vec), ufl.as_vector([0.0, 1.0])) * ds_cyl)

    # Pressure at front/rear stagnation (polygon vertices for even segments)
    p_pts  = np.array([[cx - r, cy, 0.0],
                       [cx + r, cy, 0.0]])
    p_tree = _bb_tree(msh, msh.topology.dim)
    p_cand = _ccp(p_tree, p_pts)
    p_adj  = _ccc(msh, p_cand, p_pts)
    p_cidx = np.array(
        [p_adj.array[p_adj.offsets[i]] for i in range(len(p_pts))],
        dtype=np.int32,
    )

    A_u = create_matrix(a_u); b_u = create_vector(V)
    A_p = create_matrix(a_p); b_p = create_vector(Q)

    def _ksp(A, pc_type, amg=False):
        ksp = PETSc.KSP().create(msh.comm)
        ksp.setOperators(A); ksp.setType("bcgs")
        pc = ksp.getPC(); pc.setType(pc_type)
        if amg: pc.setHYPREType("boomeramg")
        ksp.setTolerances(rtol=1e-6, atol=1e-14, max_it=500); ksp.setFromOptions()
        return ksp

    ksp_u = _ksp(A_u, "ilu")
    ksp_p = _ksp(A_p, "hypre", amg=True)

    set_bc(u0.x.petsc_vec, bcu); u0.x.scatter_forward()
    set_bc(u1.x.petsc_vec, bcu); u1.x.scatter_forward()

    t_wall0 = time.time()
    t = dt_val; step = 0; ss_res = float('nan')
    t_stop = None

    while t < T + 1e-10:
        for _ in range(5):
            A_u.zeroEntries()
            assemble_matrix(A_u, a_u, bcs=bcu); A_u.assemble()
            with b_u.localForm() as loc: loc.set(0.0)
            assemble_vector(b_u, l_u)
            apply_lifting(b_u, [a_u], bcs=[bcu])
            b_u.ghostUpdate(addv=PETSc.InsertMode.ADD,
                            mode=PETSc.ScatterMode.REVERSE)
            set_bc(b_u, bcu)
            ksp_u.setOperators(A_u, A_u); ksp_u.solve(b_u, u1.x.petsc_vec)
            u1.x.scatter_forward()
            set_bc(u1.x.petsc_vec, bcu); u1.x.scatter_forward()

            A_p.zeroEntries()
            assemble_matrix(A_p, a_p, bcs=bcp); A_p.assemble()
            with b_p.localForm() as loc: loc.set(0.0)
            assemble_vector(b_p, l_p)
            apply_lifting(b_p, [a_p], bcs=[bcp])
            b_p.ghostUpdate(addv=PETSc.InsertMode.ADD,
                            mode=PETSc.ScatterMode.REVERSE)
            set_bc(b_p, bcp)
            ksp_p.setOperators(A_p, A_p); ksp_p.solve(b_p, p1.x.petsc_vec)
            p1.x.scatter_forward()
            set_bc(p1.x.petsc_vec, bcp); p1.x.scatter_forward()

        diff_arr = u1.x.array - u0.x.array
        nd  = float(msh.comm.allreduce(np.dot(diff_arr, diff_arr), op=MPI.SUM)) ** 0.5
        n1  = float(msh.comm.allreduce(np.dot(u1.x.array, u1.x.array), op=MPI.SUM)) ** 0.5
        ss_res = nd / (dt_val * max(n1, 1e-30))

        u0.x.array[:] = u1.x.array
        t += dt_val; step += 1

        if step % 1000 == 0:
            print(f"  step={step}, t={t:.3f}, ss_res={ss_res:.2e}", flush=True)

        if ss_res < tol_ss:
            t_stop = t - dt_val
            print(f"  SS met: t={t_stop:.4f}, ss_res={ss_res:.2e}", flush=True)
            break

    if t_stop is None:
        t_stop = t - dt_val

    t_wall = time.time() - t_wall0
    print(f"  Done: {step} steps, t_stop={t_stop:.4f}, wall={t_wall:.1f}s", flush=True)

    # Volume-form forces (primary)
    F_drag = float(msh.comm.allreduce(assemble_scalar(drag_form), op=MPI.SUM))
    F_lift = float(msh.comm.allreduce(assemble_scalar(lift_form), op=MPI.SUM))
    C_D    = force_ref * F_drag
    C_L    = force_ref * F_lift

    # Surface stress forces (secondary)
    F_ds   = float(msh.comm.allreduce(assemble_scalar(surf_drag_form), op=MPI.SUM))
    F_ls   = float(msh.comm.allreduce(assemble_scalar(surf_lift_form), op=MPI.SUM))
    C_D_s  = force_ref * F_ds
    C_L_s  = force_ref * F_ls

    p_vals = p1.eval(p_pts, p_cidx)
    dp     = float(p_vals[0, 0] - p_vals[1, 0])

    print(f"  C_D_vol={C_D:.8f}  C_L_vol={C_L:.10f}  C_D_surf={C_D_s:.8f}  C_L_surf={C_L_s:.10f}  Δp={dp:.10f}",
          flush=True)

    results.append({
        "lvl": lvl_idx + 1, "res": resolution, "segs": segs, "cells": n_cells,
        "h_cyl": h_cyl, "h_D": h_D_val, "dt": dt_val, "steps": step,
        "t_stop": t_stop,
        "C_D": C_D, "C_L": C_L, "C_D_s": C_D_s, "C_L_s": C_L_s,
        "dp": dp, "wall": t_wall,
    })

# **Results**

In [ ]:
import math as _math

print("Schäfer-Turek 2D-1 — graded mesh (v6), GLS P1/P1, dt = 0.5 h_cyl")
print(f"Reference: C_D={C_D_ref}  C_L={C_L_ref}  Δp={dp_ref}")
print(f"ST96 intervals: C_D∈[5.57,5.59]  C_L∈[0.0104,0.0110]  Δp∈[0.1172,0.1176]")
print(f"f-scale = |ΔC_L| / C_D_ref (force-vector scale)")
print()

def _flag(val, lo, hi):
    return "✓" if lo <= val <= hi else "✗"

valid = [r for r in results if r is not None]
hdr = (f"{'lvl':>3} {'res':>4} {'cells':>7} {'h_cyl':>7} {'h_D':>7} {'steps':>6} {'t_stop':>7}  "
       f"{'C_D_vol':>10} {'|ΔC_D|':>8} {'✓':>2}  "
       f"{'C_D_surf':>10} {'|ΔC_D|':>8} {'✓':>2}  "
       f"{'C_L_vol':>11} {'f-sc':>7} {'✓':>2}  "
       f"{'C_L_surf':>11} {'f-sc':>7} {'✓':>2}  "
       f"{'Δp':>10} {'|ΔΔp|':>8} {'✓':>2}  "
       f"{'wall':>7}")
print(hdr)
print("-" * len(hdr))
for r in valid:
    eD_v  = abs(r["C_D"]   - C_D_ref)
    eD_s  = abs(r["C_D_s"] - C_D_ref)
    eL_fv = abs(r["C_L"]   - C_L_ref) / C_D_ref
    eL_fs = abs(r["C_L_s"] - C_L_ref) / C_D_ref
    edp   = abs(r["dp"]    - dp_ref)
    print(f"{r['lvl']:>3d} {r['res']:>4d} {r['cells']:>7d} {r['h_cyl']:>7.5f} {r['h_D']:>7.5f} "
          f"{r['steps']:>6d} {r['t_stop']:>7.3f}  "
          f"{r['C_D']:>10.6f} {eD_v:>8.5f} {_flag(r['C_D'],5.57,5.59):>2}  "
          f"{r['C_D_s']:>10.6f} {eD_s:>8.5f} {_flag(r['C_D_s'],5.57,5.59):>2}  "
          f"{r['C_L']:>11.7f} {eL_fv:>7.5f} {_flag(r['C_L'],0.0104,0.0110):>2}  "
          f"{r['C_L_s']:>11.7f} {eL_fs:>7.5f} {_flag(r['C_L_s'],0.0104,0.0110):>2}  "
          f"{r['dp']:>10.7f} {edp:>8.5f} {_flag(r['dp'],0.1172,0.1176):>2}  "
          f"{r['wall']:>6.0f}s")

# Convergence rates (log2 of absolute error ratio, L1→L2, L2→L3)
print()
print("Convergence rates (log2 |e_coarse|/|e_fine|; expected ≈1 for P1):")
rate_hdr = f"{'levels':>10}  {'C_D_vol':>7}  {'C_D_surf':>8}  {'C_L_vol(f)':>10}  {'C_L_surf(f)':>11}  {'Δp':>6}"
print(rate_hdr)
print("-" * len(rate_hdr))
for i in range(len(valid) - 1):
    a, b = valid[i], valid[i+1]
    def _rate(ea, eb):
        return f"{_math.log2(ea/eb):>+.2f}" if ea > 0 and eb > 0 else "  n/a"
    eD_v_a  = abs(a["C_D"]   - C_D_ref);  eD_v_b  = abs(b["C_D"]   - C_D_ref)
    eD_s_a  = abs(a["C_D_s"] - C_D_ref);  eD_s_b  = abs(b["C_D_s"] - C_D_ref)
    eL_a    = abs(a["C_L"]   - C_L_ref)/C_D_ref; eL_b = abs(b["C_L"] - C_L_ref)/C_D_ref
    eL_s_a  = abs(a["C_L_s"] - C_L_ref)/C_D_ref; eL_s_b = abs(b["C_L_s"] - C_L_ref)/C_D_ref
    edp_a   = abs(a["dp"]    - dp_ref);    edp_b   = abs(b["dp"]    - dp_ref)
    tag = f"L{a['lvl']}→L{b['lvl']}"
    print(f"{tag:>10}  {_rate(eD_v_a,eD_v_b):>7}  {_rate(eD_s_a,eD_s_b):>8}  "
          f"{_rate(eL_a,eL_b):>10}  {_rate(eL_s_a,eL_s_b):>11}  {_rate(edp_a,edp_b):>6}")

# **Discussion**

The graded mesh (v6) extends the fine zone to dist_max = 3D = 0.30 m into the wake, increasing cell count to 2751 (L1) and 10716 (L2) versus 1236 and 4652 on the ungraded mesh. Wall time at L2 is 7168 s (≈ 2 h); L3 (resolution=128) is running and expected to require ≈ 8 h.

**C_D (volume form)** converges at rate ≈ 1.0 (L1 error 0.40% → L2 error 0.20%), consistent with first-order P1 elements under mesh halving. The L2 value 5.5908 is 0.079‰ above the ST96 upper bound [5.57, 5.59]; L3 is expected to bring it within the interval.

**C_L (volume form)** at L1 is −0.0074 (wrong sign). The cylinder centre (y = 0.2) is only 0.005 m from the channel mid-line (y = 0.205), making C_L a near-cancellation quantity; the coarse-level graded mesh still cannot resolve the wake asymmetry. At L2 the extended fine zone captures the wake and C_L = 0.01077 (✓ in [0.0104, 0.0110]); the force-vector scale error drops from 0.00324 to 0.00003. The sign change between levels means the L1→L2 rate is not interpretable as a convergence rate; the monotone regime begins at L2 or higher.

**Pressure drop Δp** does not converge monotonically: L1 overshoots (0.11811) and L2 undershoots (0.11653) the reference 0.11752. Non-monotone pressure convergence is characteristic of GLS-stabilized P1/P1 on coarse meshes where the stabilization term introduces a mesh-dependent pressure perturbation. Both values fall outside the ST96 interval [0.1172, 0.1176].

**Surface stress (secondary, this notebook only):** C_D_surf = 5.393 (L1) and 5.474 (L2), errors 3.35% and 1.89% — larger than the volume form (0.40%, 0.20%). C_L_surf = −0.0815 (L1) and 0.0020 (L2), not within the ST96 interval at either level. The volume form is more accurate because it avoids differentiating the P1 velocity, which amplifies surface-traction errors.